<a href="https://colab.research.google.com/github/dimitarpg13/agentic_architectures_and_design_patterns/blob/main/notebooks/model_evaluation/mlflow/google_colab/Mlflow_experiment_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Complete MLflow Setup for Google Colab
1. Installation and Basic Setup

In [1]:
# Install required packages
!pip install mlflow --quiet
!pip install pyngrok --quiet  # For exposing MLflow UI
!pip install mlflow[extras] --quiet  # Optional: additional integrations

import mlflow
import os
from pathlib import Path
import subprocess
import threading
import time

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.2/779.2 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 11.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━

2. Google Drive Integration (Recommended for Persistence)

In [2]:
from google.colab import drive

# Mount Google Drive for persistent storage
drive.mount('/content/drive')

# Set up MLflow tracking directory in Google Drive
MLFLOW_TRACKING_DIR = '/content/drive/MyDrive/mlflow_experiments'
Path(MLFLOW_TRACKING_DIR).mkdir(parents=True, exist_ok=True)

# Configure MLflow tracking URI
mlflow.set_tracking_uri(f'file://{MLFLOW_TRACKING_DIR}')
print(f"MLflow tracking URI set to: {mlflow.get_tracking_uri()}")

Mounted at /content/drive
MLflow tracking URI set to: file:///content/drive/MyDrive/mlflow_experiments


3. MLflow UI Setup with Ngrok

In [13]:
from pyngrok import ngrok, conf
import getpass

def setup_mlflow_ui(tracking_uri=None, port=5000):
    """
    Set up MLflow UI with ngrok tunnel for Colab access
    """
    # Set up ngrok authentication if needed
    # You can get a free auth token from https://dashboard.ngrok.com/
    print("Enter your ngrok authtoken (optional, press Enter to skip):")
    print("Get one free at: https://dashboard.ngrok.com/signup")
    auth_token = getpass.getpass()

    if auth_token:
        ngrok.set_auth_token(auth_token)

    # Kill any existing MLflow server processes
    subprocess.run(["pkill", "-f", "mlflow.server"], capture_output=True)
    time.sleep(2)

    # Start MLflow server
    tracking_uri = tracking_uri or mlflow.get_tracking_uri()
    mlflow_cmd = [
        "mlflow", "server",
        "--backend-store-uri", tracking_uri,
        "--default-artifact-root", tracking_uri,
        "--host", "0.0.0.0",
        "--port", str(port),
    ]

    # Run MLflow server in background
    mlflow_process = subprocess.Popen(
        mlflow_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    # Wait for server to start
    time.sleep(5)

    # Create ngrok tunnel
    try:
        # Kill any existing tunnels
        ngrok.kill()
        time.sleep(2)

        # Create new tunnel
        public_url = ngrok.connect(port, "http")
        print(f"\n✅ MLflow UI available at: {public_url}")
        print(f"📊 Tracking URI: {tracking_uri}")
        return public_url, mlflow_process
    except Exception as e:
        print(f"Error setting up ngrok: {e}")
        print("Falling back to local access only")
        return f"http://localhost:{port}", mlflow_process

# Start MLflow UI
# mlflow_url, mlflow_process = setup_mlflow_ui(MLFLOW_TRACKING_DIR)

Enter your ngrok authtoken (optional, press Enter to skip):
Get one free at: https://dashboard.ngrok.com/signup
··········

✅ MLflow UI available at: NgrokTunnel: "https://ashely-nonviviparous-lillian.ngrok-free.dev" -> "http://localhost:5000"
📊 Tracking URI: /content/drive/MyDrive/mlflow_experiments


4. Alternative: Using Localtunnel (No Auth Required)

In [3]:
# Alternative to ngrok - doesn't require authentication
!npm install -g localtunnel

def setup_mlflow_with_localtunnel(tracking_uri=None, port=5000):
    """
    Set up MLflow UI with localtunnel (no auth required)
    """
    import subprocess
    import threading
    import time

    tracking_uri = tracking_uri or mlflow.get_tracking_uri()

    # Kill existing MLflow servers
    subprocess.run(["pkill", "-f", "mlflow.server"], capture_output=True)
    time.sleep(2)

    # Start MLflow server
    mlflow_cmd = f"mlflow server --backend-store-uri {tracking_uri} --host 0.0.0.0 --port {port}"
    mlflow_thread = threading.Thread(
        target=lambda: subprocess.run(mlflow_cmd, shell=True)
    )
    mlflow_thread.daemon = True
    mlflow_thread.start()
    time.sleep(5)

    # Start localtunnel
    lt_process = subprocess.Popen(
        f"lt --port {port} --local-host localhost",
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    # Get URL from localtunnel output
    for line in lt_process.stdout:
        if "your url is" in line.lower():
            url = line.split("https://")[-1].strip()
            print(f"\n✅ MLflow UI available at: https://{url}")
            print(f"📊 Tracking URI: {tracking_uri}")
            break

    return lt_process

# Uncomment to use localtunnel instead
# lt_process = setup_mlflow_with_localtunnel(MLFLOW_TRACKING_DIR)

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 4s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼npm notice
npm notice New major version of npm available! 10.8.2 -> 11.7.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.7.0
npm notice To update run: npm install -g npm@11.7.0
npm notice
⠼
✅ MLflow UI available at: https://empty-donuts-dress.loca.lt
📊 Tracking URI: /content/drive/MyDrive/mlflow_experiments


5. Production-Ready MLflow Configuration Class

In [16]:
import json
from datetime import datetime
from typing import Optional, Dict, Any
import pandas as pd

class ColabMLflowManager:
    """
    Complete MLflow manager for Google Colab environments
    """

    def __init__(
        self,
        experiment_name: str,
        use_drive: bool = True,
        drive_folder: str = 'mlflow_experiments',
        auto_start_ui: bool = True
    ):
        self.experiment_name = experiment_name
        self.use_drive = use_drive
        self.drive_folder = drive_folder
        self.mlflow_process = None
        self.public_url = None

        # Setup storage
        self._setup_storage()

        # Configure MLflow
        mlflow.set_tracking_uri(self.tracking_uri)

        # Create or get experiment
        self.experiment = mlflow.set_experiment(experiment_name)

        # Start UI if requested
        if auto_start_ui:
            self.start_ui()

    def _setup_storage(self):
        """Configure storage backend"""
        if self.use_drive:
            try:
                from google.colab import drive
                drive.mount('/content/drive', force_remount=False)
                self.base_path = f'/content/drive/MyDrive/{self.drive_folder}'
            except:
                print("⚠️ Google Drive mount failed, using local storage")
                self.base_path = f'/content/{self.drive_folder}'
        else:
            self.base_path = f'/content/{self.drive_folder}'

        # Create directory
        Path(self.base_path).mkdir(parents=True, exist_ok=True)
        self.tracking_uri = f'file://{self.base_path}'
        print(f"📁 MLflow storage: {self.base_path}")

    def start_ui(self, port: int = 5000, use_ngrok: bool = True):
        """Start MLflow UI with public access"""
        # Kill existing servers
        subprocess.run(["pkill", "-f", "mlflow.server"], capture_output=True)
        time.sleep(2)

        # Start MLflow server
        mlflow_cmd = [
            "mlflow", "server",
            "--backend-store-uri", self.tracking_uri,
            "--default-artifact-root", self.tracking_uri,
            "--host", "0.0.0.0",
            "--port", str(port),
            "--serve-artifacts"
        ]

        self.mlflow_process = subprocess.Popen(
            mlflow_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE
        )

        print("⏳ Starting MLflow server...")
        time.sleep(5)
        config = {
          "addr": port,  # Your local port
          "proto": "http",
          "host_header": f"localhost:{str(port)}"  # Custom host header
        }

        if use_ngrok:

            try:
                from pyngrok import ngrok
                auth_token = getpass.getpass()

                if auth_token:
                    ngrok.set_auth_token(auth_token)
                ngrok.kill()
                time.sleep(1)
                self.public_url = ngrok.connect(**config)
                print(f"\n✅ MLflow UI: {self.public_url}")
                print(f"📊 Experiment: {self.experiment_name}")
            except Exception as e:
                print(f"⚠️ Ngrok setup failed: {e}")
                self._try_localtunnel(port)
        else:
            self._try_localtunnel(port)

    def _try_localtunnel(self, port: int):
        """Fallback to localtunnel"""
        try:
            subprocess.run(["npm", "install", "-g", "localtunnel"],
                         capture_output=True, check=True)

            lt_process = subprocess.Popen(
                f"lt --port {port} --local-host localhost",
                shell=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )

            for line in lt_process.stdout:
                if "your url is" in line.lower():
                    self.public_url = line.split("https://")[-1].strip()
                    print(f"\n✅ MLflow UI: https://{self.public_url}")
                    break
        except Exception as e:
            print(f"⚠️ Localtunnel failed: {e}")
            print(f"📍 MLflow running locally on port {port}")

    def log_metrics_batch(self, metrics: Dict[str, Any], step: int = 0):
        """Log multiple metrics at once"""
        for key, value in metrics.items():
            mlflow.log_metric(key, value, step=step)

    def log_params_batch(self, params: Dict[str, Any]):
        """Log multiple parameters at once"""
        for key, value in params.items():
            mlflow.log_param(key, value)

    def log_model_with_metadata(
        self,
        model,
        artifact_path: str,
        model_type: str = "sklearn",
        metadata: Optional[Dict] = None
    ):
        """Log model with comprehensive metadata"""
        # Log model
        if model_type == "sklearn":
            mlflow.sklearn.log_model(model, artifact_path)
        elif model_type == "pytorch":
            mlflow.pytorch.log_model(model, artifact_path)
        elif model_type == "tensorflow":
            mlflow.tensorflow.log_model(model, artifact_path)
        else:
            mlflow.pyfunc.log_model(artifact_path, python_model=model)

        # Log metadata
        if metadata:
            metadata['timestamp'] = datetime.now().isoformat()
            metadata['model_type'] = model_type

            with open('/tmp/model_metadata.json', 'w') as f:
                json.dump(metadata, f, indent=2)
            mlflow.log_artifact('/tmp/model_metadata.json')

    def cleanup(self):
        """Clean up resources"""
        if self.mlflow_process:
            self.mlflow_process.terminate()
            self.mlflow_process.wait()

        if self.public_url:
            try:
                from pyngrok import ngrok
                ngrok.kill()
            except:
                pass

    def get_experiment_info(self) -> pd.DataFrame:
        """Get experiment runs as DataFrame"""
        runs = mlflow.search_runs(
            experiment_ids=[self.experiment.experiment_id]
        )
        return runs

6. Example Usage

In [17]:
# Initialize MLflow manager
mlflow_manager = ColabMLflowManager(
    experiment_name="colab_experiment",
    use_drive=True,
    auto_start_ui=True
)

# Example training with MLflow logging
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Generate sample data
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Start MLflow run
with mlflow.start_run(run_name="rf_classifier_run"):
    # Log parameters
    params = {
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_split": 5,
        "random_state": 42
    }
    mlflow_manager.log_params_batch(params)

    # Train model
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Log metrics
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred, average='weighted'),
        "train_samples": len(X_train),
        "test_samples": len(X_test)
    }
    mlflow_manager.log_metrics_batch(metrics)

    # Log model with metadata
    mlflow_manager.log_model_with_metadata(
        model=model,
        artifact_path="random_forest",
        model_type="sklearn",
        metadata={
            "feature_importance": model.feature_importances_.tolist(),
            "n_features": X.shape[1]
        }
    )

    # Log additional artifacts
    mlflow.log_dict(params, "hyperparameters.json")
    mlflow.log_dict(metrics, "metrics.json")

    print(f"✅ Run completed!")
    print(f"📊 Metrics: {metrics}")

# View experiment results
runs_df = mlflow_manager.get_experiment_info()
print("\n📈 Experiment Runs:")
print(runs_df[['run_id', 'status', 'metrics.accuracy', 'metrics.f1_score']].head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 MLflow storage: /content/drive/MyDrive/mlflow_experiments
⏳ Starting MLflow server...
··········

✅ MLflow UI: NgrokTunnel: "https://ashely-nonviviparous-lillian.ngrok-free.dev" -> "http://localhost:5000"
📊 Experiment: colab_experiment


2026/01/11 07:59:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Run completed!
📊 Metrics: {'accuracy': 0.9, 'f1_score': 0.8997596153846155, 'train_samples': 800, 'test_samples': 200}

📈 Experiment Runs:
                             run_id    status  metrics.accuracy  \
0  14c5753404af4904af1ef32b469e7d49  FINISHED             0.900   
1  0fc48ace86114ad68519775d850a4ecb    FAILED             0.850   
2  3116836dcd0145f081c14bee614977ea  FINISHED             0.905   
3  daebe3a43bef41edbf63c0ce1bd3f36e  FINISHED             0.875   

   metrics.f1_score  
0          0.899760  
1          0.850000  
2          0.904850  
3          0.874803  
